# 02 - LLM Knowledge Extraction

**Task 1, Fase 2:** Mengekstrak entitas dan relasi dari chunks UU ITE menggunakan Gemini API, lalu memvalidasi dan mendeduplikasi hasilnya.

**Pipeline:** `Chunks → LLM Extract → Validate → Deduplicate`

In [10]:
# === Setup ===
import sys
import json
import os
from pathlib import Path
from dotenv import load_dotenv

PROJECT_ROOT = Path(os.getcwd()).parent.parent if 'notebooks' in str(Path(os.getcwd())) else Path(os.getcwd())
sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)
load_dotenv()

GEMINI_API_KEY = os.getenv('GEMINI_API_KEY')
print(f'Project root: {PROJECT_ROOT}')
print(f'API Key: {GEMINI_API_KEY[:10]}...' if GEMINI_API_KEY else 'ERROR: No API key found!')

Project root: d:\TA\llm-driven-legal-kg-visualization
API Key: AIzaSyB38G...


## Step 0: Configure PROMPT_ID

Set `PROMPT_ID` to select which prompt to use from the **KG_EXTRACTION_PROMPT** sheet in Google Sheets.

The sheet should have columns: `PROMPT_ID`, `SYSTEM_PROMPT`, `USER_PROMPT`

In [11]:
# === Configure PROMPT_ID ===
# Change this to select a different prompt version from Google Sheets
PROMPT_ID = "PROMPT_3"  # e.g., "PROMPT_1", "PROMPT_2"

print(f'Selected PROMPT_ID: {PROMPT_ID}')

Selected PROMPT_ID: PROMPT_3


In [12]:
# === Fetch Prompt from Google Sheets ===
from modules.prompt_fetcher import fetch_kg_extraction_prompt

prompt_data = fetch_kg_extraction_prompt(PROMPT_ID)

SYSTEM_PROMPT = prompt_data['SYSTEM_PROMPT']
USER_PROMPT_TEMPLATE = prompt_data['USER_PROMPT']

print(f'✅ Loaded prompt: {PROMPT_ID}')
print(f'System prompt length: {len(SYSTEM_PROMPT)} chars')
print(f'User prompt template length: {len(USER_PROMPT_TEMPLATE)} chars')
print(f'\n--- System Prompt Preview (first 300 chars) ---')
print(SYSTEM_PROMPT[:300])
print(f'\n--- User Prompt Template Preview ---')
print(USER_PROMPT_TEMPLATE[:200])

2026-04-19 20:54:56,003 - INFO - Fetching prompt 'PROMPT_3' from sheet 'KG_EXTRACTION_PROMPT'...
2026-04-19 20:54:56,007 - INFO - Retrieving worksheet 'KG_EXTRACTION_PROMPT' from spreadsheet ID '1oN5kMN_OI8WyITAQgJ3-S_0GlzraXug8p2tMKSmq7u0'...
2026-04-19 20:54:58,294 - INFO - Successfully loaded 4 rows from worksheet 'KG_EXTRACTION_PROMPT'.
2026-04-19 20:54:58,321 - INFO - Loaded prompt 'PROMPT_3': ['PROMPT_ID', 'SYSTEM_PROMPT', 'USER_PROMPT', 'NOTES']


✅ Loaded prompt: PROMPT_3
System prompt length: 5135 chars
User prompt template length: 148 chars

--- System Prompt Preview (first 300 chars) ---
You are a Knowledge Graph extractor for Indonesian legal documents.
Given a chunk of legal text, extract entities (nodes) and relationships (edges) according to the ontology below.

## Valid Node Types

| Type | Description | Example Labels |
|------|-------------|----------------|
| Regulasi | The 

--- User Prompt Template Preview ---
Extract all entities and relationships from the following Indonesian legal text.
Document ID: {document_id}

<LEGAL_TEXT>
{chunk_text}
</LEGAL_TEXT>


## Step 1: Load Chunks

In [13]:
# Load chunks dari Fase 1
CHUNKS_PATH = 'data/chunks/UU_11_2008_chunks.json'

with open(CHUNKS_PATH, 'r', encoding='utf-8') as f:
    chunks_data = json.load(f)

chunks = chunks_data['chunks']
print(f'Document: {chunks_data["document_id"]}')
print(f'Total chunks: {len(chunks)}')
print(f'Token stats: {chunks_data.get("stats", {})}')

Document: UU_11_2008
Total chunks: 15
Token stats: {'min_tokens': 89, 'max_tokens': 2103, 'avg_tokens': 968.0}


## Step 2: Extract KG Triples with Gemini

Mengirim chunks ke Gemini API dengan system prompt ontologi. Output: nodes (entities) dan edges (relations).

⚠️ **Proses ini membutuhkan API calls dan memakan waktu beberapa menit.**

Uses prompt from Google Sheets (PROMPT_ID configured above).

In [14]:
from pipeline.transform.llm_extractor import extract_all_triples

# Extract triples using prompt from Google Sheets
triples_path = extract_all_triples(
    chunks_path=CHUNKS_PATH,
    output_dir='data/triples',
    api_key=GEMINI_API_KEY,
    model_name='gemini-2.5-flash',
    batch_size=3,
    max_retries=3,
    delay_between_calls=2.0,
    system_prompt=SYSTEM_PROMPT,
    user_prompt_template=USER_PROMPT_TEMPLATE,
    prompt_id=PROMPT_ID,
)

# Load results
with open(triples_path, 'r', encoding='utf-8') as f:
    triples_data = json.load(f)

print(f'\n=== Extraction Summary ===')
print(f'Prompt ID: {PROMPT_ID}')
print(f'Total Nodes: {triples_data["total_nodes"]}')
print(f'Total Edges: {triples_data["total_edges"]}')
print(f'Errors: {len(triples_data.get("errors", []))}')
print(f'Saved to: {triples_path}')

Extracting UU_11_2008:   0%|          | 0/5 [00:00<?, ?it/s]

Extracting UU_11_2008: 100%|██████████| 5/5 [10:17<00:00, 123.58s/it]


=== Extraction Summary ===
Prompt ID: PROMPT_3
Total Nodes: 455
Total Edges: 671
Errors: 0
Saved to: data/triples\UU_11_2008_PROMPT_3_triples.json


In [15]:
# Preview nodes by type
from collections import Counter

node_types = Counter(n['type'] for n in triples_data['nodes'])
edge_types = Counter(e['type'] for e in triples_data['edges'])

print('=== Node Types ===')
for t, count in node_types.most_common():
    print(f'  {t:20s}: {count}')

print(f'\n=== Edge Types ===')
for t, count in edge_types.most_common():
    print(f'  {t:20s}: {count}')

# Sample nodes
print(f'\n=== Sample Nodes (first 10) ===')
for n in triples_data['nodes'][:10]:
    print(f'  [{n["type"]}] {n["label"]}')

=== Node Types ===
  PerbuatanHukum      : 136
  Ayat                : 123
  Pasal               : 67
  EntitasHukum        : 64
  KonsepHukum         : 29
  Sanksi              : 16
  Bab                 : 13
  Regulasi            : 5
  Bagian              : 2

=== Edge Types ===
  BERLAKU_UNTUK       : 180
  MENGATUR            : 136
  MEMILIKI_AYAT       : 123
  MERUJUK             : 121
  MEMUAT              : 69
  MENDEFINISIKAN      : 23
  MENETAPKAN_SANKSI   : 19

=== Sample Nodes (first 10) ===
  [Regulasi] Undang-Undang tentang Informasi dan Transaksi Elektronik
  [Bab] BAB I
  [Pasal] Pasal 1
  [KonsepHukum] Informasi Elektronik
  [KonsepHukum] Transaksi Elektronik
  [KonsepHukum] Teknologi Informasi
  [KonsepHukum] Dokumen Elektronik
  [KonsepHukum] Sistem Elektronik
  [KonsepHukum] Penyelenggaraan Sistem Elektronik
  [KonsepHukum] Jaringan Sistem Elektronik


## Step 3: Validate Triples

In [16]:
from pipeline.transform.validator import validate_triples_file

validated_path = validate_triples_file(
    input_path=triples_path,
    output_dir='data/validated',
    strict=False,  # Set True for stricter edge type checking
    prompt_id=PROMPT_ID,
)

with open(validated_path, 'r', encoding='utf-8') as f:
    validated_data = json.load(f)

print(f'=== Validation Summary ===')
print(f'Nodes: {validated_data["total_nodes"]} (removed {validated_data["removed_nodes"]})')
print(f'Edges: {validated_data["total_edges"]} (removed {validated_data["removed_edges"]})')
print(f'Saved to: {validated_path}')

# Show errors if any
error_log = validated_path.replace('_triples.json', '_errors.log')
if os.path.exists(error_log):
    with open(error_log, 'r', encoding='utf-8') as f:
        errors = f.read()
    print(f'\n=== Errors ({errors.count(chr(10))} lines) ===')
    print(errors[:1000])

=== Validation Summary ===
Nodes: 455 (removed 0)
Edges: 671 (removed 0)
Saved to: data/validated\UU_11_2008_PROMPT_3_triples.json

=== Errors (5 lines) ===
Validation Errors for UU_11_2008
Total errors: 1

1. Edge target not found: Ayat_20_2 -[MENGATUR]-> PerbuatanHukum_melakukan_persetujuan_penawaran_Transaksi_Elektronik_dengan_pernyataan_penerimaan_ secara_elektronik



## Step 4: Deduplicate Entities

In [17]:
from pipeline.transform.deduplicator import deduplicate_triples_file

deduped_path = deduplicate_triples_file(
    input_path=validated_path,
    output_dir='data/deduped',
    similarity_threshold=0.85,
    prompt_id=PROMPT_ID,
)

with open(deduped_path, 'r', encoding='utf-8') as f:
    deduped_data = json.load(f)

print(f'=== Deduplication Summary ===')
print(f'Nodes: {deduped_data["nodes_before"]} → {deduped_data["total_nodes"]} (merged {deduped_data["nodes_merged"]})')
print(f'Edges: {deduped_data["edges_before"]} → {deduped_data["total_edges"]}')
print(f'Saved to: {deduped_path}')

=== Deduplication Summary ===
Nodes: 455 → 406 (merged 22)
Edges: 671 → 653
Saved to: data/deduped\UU_11_2008_PROMPT_3_triples.json


In [18]:
# Final KG overview
print('=== Final Knowledge Graph ===')
final_node_types = Counter(n['type'] for n in deduped_data['nodes'])
final_edge_types = Counter(e['type'] for e in deduped_data['edges'])

print(f'\nNode Types ({deduped_data["total_nodes"]} total):')
for t, count in final_node_types.most_common():
    print(f'  {t:20s}: {count}')

print(f'\nEdge Types ({deduped_data["total_edges"]} total):')
for t, count in final_edge_types.most_common():
    print(f'  {t:20s}: {count}')

print(f'\n=== Sample Triples ===')
node_map = {n['id']: n['label'] for n in deduped_data['nodes']}
for e in deduped_data['edges'][:15]:
    src = node_map.get(e.get('source_id', e.get('source', '')), e.get('source_id', '?'))
    tgt = node_map.get(e.get('target_id', e.get('target', '')), e.get('target_id', '?'))
    print(f'  {src} —[{e["type"]}]→ {tgt}')

=== Final Knowledge Graph ===

Node Types (406 total):
  PerbuatanHukum      : 133
  Ayat                : 105
  Pasal               : 54
  EntitasHukum        : 54
  KonsepHukum         : 26
  Sanksi              : 16
  Bab                 : 13
  Regulasi            : 3
  Bagian              : 2

Edge Types (653 total):
  BERLAKU_UNTUK       : 180
  MENGATUR            : 136
  MERUJUK             : 121
  MEMILIKI_AYAT       : 105
  MEMUAT              : 69
  MENDEFINISIKAN      : 23
  MENETAPKAN_SANKSI   : 19

=== Sample Triples ===
  Undang-Undang tentang Informasi dan Transaksi Elektronik —[MEMUAT]→ BAB I
  BAB I —[MEMUAT]→ Pasal 1
  BAB I —[MEMUAT]→ Pasal 2
  Pasal 1 —[MENDEFINISIKAN]→ Informasi Elektronik
  Pasal 1 —[MENDEFINISIKAN]→ Transaksi Elektronik
  Pasal 1 —[MENDEFINISIKAN]→ Teknologi Informasi
  Pasal 1 —[MENDEFINISIKAN]→ Dokumen Elektronik
  Pasal 1 —[MENDEFINISIKAN]→ Sistem Elektronik
  Pasal 1 —[MENDEFINISIKAN]→ Penyelenggaraan Sistem Elektronik
  Pasal 1 —[MENDEFINISI

## Summary

Fase 2 selesai! Output tersimpan di:
- `data/triples/{doc_id}_{prompt_id}_triples.json` — Raw LLM extraction
- `data/validated/{doc_id}_{prompt_id}_triples.json` — Validated (invalid types removed)
- `data/deduped/{doc_id}_{prompt_id}_triples.json` — Deduplicated (final)

**Prompt used:** Loaded from Google Sheets sheet `KG_EXTRACTION_PROMPT` with PROMPT_ID configured above.

**Next:** Run `03_neo4j_ingestion.ipynb` to load into Neo4j.